In [1]:
#!/usr/bin/env python3
"""
Scrape incoming freshmen major-junior stats from HockeyDB.

Tailored to roster columns:
['Current Team','Last_Name','First_Name','No','Position','Yr','Ht','Wt','DOB',
 'Hometown','Height_Inches','Draft_Year','NHL_Team','D_Round','Last Team',
 'League','City','State_Province','Country']

Install:
  pip install requests beautifulsoup4 lxml pandas rapidfuzz tqdm python-dateutil

Example:
  python hdb_freshmen_majorjunior.py \
    --roster roster.csv \
    --out freshmen_majorjunior.csv \
    --freshman-values Fr FR Freshman \
    --limit 5 \
    --leagues OHL WHL QMJHL

Notes:
- Review hockeydb.com terms/robots.txt and comply.
- Adds polite delays + caching to be a good netizen.
"""

from __future__ import annotations
import argparse
import json
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, List, Dict

import pandas as pd
import requests
from bs4 import BeautifulSoup
from rapidfuzz import fuzz, process
from tqdm import tqdm
from dateutil import parser as dtparse

DUCKDUCKGO_HTML = "https://duckduckgo.com/html/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/125.0.0.0 Safari/537.36",
}

CACHE_DIR = Path(".hdb_cache")
CACHE_DIR.mkdir(exist_ok=True)

def _sleep_jitter(min_s=1.2, max_s=2.8):
    time.sleep(random.uniform(min_s, max_s))

def _cache_path(name: str) -> Path:
    safe = re.sub(r"[^a-zA-Z0-9_.-]+", "_", name)
    return CACHE_DIR / f"{safe}.json"

def _load_cache(name: str) -> Optional[dict]:
    p = _cache_path(name)
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return None
    return None

def _save_cache(name: str, data: dict) -> None:
    p = _cache_path(name)
    p.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def ddg_search_site(name: str) -> List[str]:
    """Search for HockeyDB player pages via DuckDuckGo HTML."""
    params = {"q": f"site:hockeydb.com {name}"}
    resp = requests.get(DUCKDUCKGO_HTML, params=params, headers=HEADERS, timeout=25)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")
    links = []
    for a in soup.select("a.result__a"):
        href = a.get("href", "")
        if "hockeydb.com" in href and ("pdisplay" in href or "player.php?pid=" in href):
            links.append(href)
    # de-dup while preserving order
    seen, uniq = set(), []
    for u in links:
        if u not in seen:
            uniq.append(u); seen.add(u)
    return uniq

def fetch(url: str) -> str:
    _sleep_jitter(1.0, 2.2)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def parse_player_bio(soup: BeautifulSoup) -> Dict[str, str]:
    """Grab Birth Year and Position from page text (tolerant)."""
    text = " ".join(s.strip() for s in soup.get_text(" ").split())
    out = {}
    mpos = re.search(r"\b(Position|Pos)\s*[:\-]\s*([A-Za-z/]+)\b", text, re.I)
    if mpos:
        out["position"] = mpos.group(2).upper()
    myear = re.search(r"\bBorn\b.*?\b(\d{4})\b", text, re.I)
    if myear:
        out["birth_year"] = myear.group(1)
    return out

def parse_stats_table(soup: BeautifulSoup) -> pd.DataFrame:
    """Find the main stats table (Season, Team, League, GP/G/A/Pts...)."""
    candidates = []
    for tbl in soup.find_all("table"):
        heads = [c.get_text(strip=True) for c in tbl.find_all("th")]
        head_line = " | ".join(h.upper() for h in heads)
        score = sum(int(k in head_line) for k in
                    ["SEASON","YEAR","TEAM","LEAGUE","GP","G","A","PTS","PIM"])
        candidates.append((score, tbl))
    if not candidates:
        return pd.DataFrame()

    _, best_tbl = max(candidates, key=lambda x: x[0])
    rows = []
    col_names = [c.get_text(strip=True) for c in best_tbl.find_all("th")]
    for tr in best_tbl.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue
        vals = [td.get_text(strip=True) for td in tds]
        rows.append(vals)
    df = pd.DataFrame(rows)
    if col_names and len(col_names) == df.shape[1]:
        df.columns = col_names
    else:
        # best-effort
        base = ["Season","Team","League","GP","G","A","Pts","PIM"]
        df = df.rename(columns={i: base[i] for i in range(min(len(base), df.shape[1]))})
    return df

@dataclass
class PlayerHint:
    birth_year: Optional[int] = None
    last_team: Optional[str] = None
    position: Optional[str] = None

def pick_best_link(name: str, links: List[str], hint: PlayerHint) -> Optional[str]:
    """Rank candidates by fuzzy name + optional hint checks (birth year / pos / last team)."""
    if not links:
        return None
    scored = []
    for url in links:
        slug = url.split("/")[-1]
        slug = re.sub(r"[-_]", " ", slug)
        s = fuzz.token_set_ratio(name, slug)
        if "pdisplay" in url or "player.php?pid=" in url:
            s += 5
        scored.append((s, url))
    scored.sort(reverse=True)
    top = [u for _, u in scored[:5]]

    best, best_score = None, -1
    for url in top:
        try:
            html = fetch(url)
        except Exception:
            continue
        soup = BeautifulSoup(html, "lxml")
        bio = parse_player_bio(soup)
        stat_df = parse_stats_table(soup)

        score = 0
        score += fuzz.token_set_ratio(name, soup.title.get_text() if soup.title else "")
        if hint.birth_year and bio.get("birth_year"):
            score += 12 if str(hint.birth_year) == str(bio["birth_year"]) else 0
        if hint.position and bio.get("position"):
            if hint.position[0:1].upper() == bio["position"][0:1].upper():
                score += 6
        if hint.last_team and not stat_df.empty and "Team" in stat_df.columns:
            match = process.extractOne(
                str(hint.last_team),
                stat_df["Team"].astype(str).tolist(),
                scorer=fuzz.token_set_ratio
            )
            if match and match[1] >= 85:
                score += 8
        if score > best_score:
            best_score, best = score, url
    return best or scored[0][1]

def normalize_league(s: str) -> str:
    return (s or "").upper().replace(".", "").strip()

def filter_major_junior(df: pd.DataFrame, leagues_whitelist: List[str]) -> pd.DataFrame:
    """Keep only rows whose League is in the whitelist (e.g., OHL/WHL/QMJHL)."""
    if df.empty:
        return df
    colmap = {c.lower(): c for c in df.columns}
    lg = colmap.get("league") or colmap.get("lg")
    if not lg:
        return pd.DataFrame()
    tmp = df.copy()
    tmp[lg] = tmp[lg].astype(str).map(normalize_league)
    wl = {normalize_league(x) for x in leagues_whitelist}
    keep = tmp[tmp[lg].isin(wl)].copy()
    # remove playoffs/totals rows by sniffing Season col if present
    sc = colmap.get("season") or colmap.get("year")
    if sc:
        keep = keep[~keep[sc].str.contains("Playoff|Total|Totals|Regular", case=False, na=False)]
    return keep

def scrape_player(name: str, hint: PlayerHint, leagues_whitelist: List[str]) -> Dict[str, any]:
    """
    Returns:
      { name, url, birth_year, position, table:[...], error? }
      where table is already filtered to leagues_whitelist.
    """
    cache_key = f"mj::{name}::{','.join(sorted(leagues_whitelist))}"
    cached = _load_cache(cache_key)
    if cached:
        return cached

    links = ddg_search_site(name)
    if not links:
        out = {"name": name, "url": None, "table": [], "error": "no_link_found"}
        _save_cache(cache_key, out)
        return out

    url = pick_best_link(name, links, hint)
    if not url:
        out = {"name": name, "url": None, "table": [], "error": "no_suitable_link"}
        _save_cache(cache_key, out)
        return out

    html = fetch(url)
    soup = BeautifulSoup(html, "lxml")
    bio = parse_player_bio(soup)
    df = parse_stats_table(soup)

    out = {"name": name, "url": url, "birth_year": bio.get("birth_year"), "position": bio.get("position")}
    if df.empty:
        out["table"] = []
        out["error"] = "no_stats_table"
        _save_cache(cache_key, out)
        return out

    # numeric cleanup best-effort
    for col in ["GP","G","A","Pts","PIM","+/-","PPG","SHG","GWG"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace("", 0), errors="coerce")

    mj = filter_major_junior(df, leagues_whitelist)
    out["table"] = mj.to_dict(orient="records")
    if not out["table"]:
        out["note"] = "no_rows_in_whitelist"
    _save_cache(cache_key, out)
    return out

def parse_birth_year(dob_val: str) -> Optional[int]:
    if not dob_val or pd.isna(dob_val):
        return None
    try:
        # Accept YYYY-MM-DD, MM/DD/YYYY, etc.
        return dtparse.parse(str(dob_val), fuzzy=True).year
    except Exception:
        # If it's just YYYY, that’s fine
        m = re.fullmatch(r"\s*(\d{4})\s*", str(dob_val))
        return int(m.group(1)) if m else None

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--roster", required=True, help="Path to roster CSV")
    ap.add_argument("--out", default="freshmen_majorjunior.csv", help="Output CSV")
    ap.add_argument("--freshman-col", default="Yr", help="Column indicating class year")
    ap.add_argument("--freshman-values", nargs="+", default=["Fr","FR","Freshman"],
                    help="Values indicating freshman")
    ap.add_argument("--first-name-col", default="First_Name")
    ap.add_argument("--last-name-col", default="Last_Name")
    ap.add_argument("--position-col", default="Position")
    ap.add_argument("--dob-col", default="DOB")
    ap.add_argument("--last-team-col", default="Last Team")
    ap.add_argument("--leagues", nargs="+", default=["OHL","WHL","QMJHL"],
                    help="Whitelist of major junior leagues to keep")
    ap.add_argument("--limit", type=int, default=None,
                    help="Max number of players to scrape (testing)")
    args = ap.parse_args()

    df = pd.read_csv(args.roster)
    # Filter to freshmen
    mask = df[args.freshman_col].astype(str).str.strip().isin(args.freshman_values)
    freshmen = df[mask].copy()

    if freshmen.empty:
        print("No freshmen found with given filters.")
        return

    if args.limit is not None:
        freshmen = freshmen.iloc[:args.limit].copy()

    results = []
    for _, row in tqdm(freshmen.iterrows(), total=len(freshmen), desc="Scraping freshmen"):
        first = str(row.get(args.first_name_col, "")).strip()
        last = str(row.get(args.last_name_col, "")).strip()
        if not first or not last:
            results.append({"Name": f"{first} {last}".strip(), "Error": "missing_name"})
            continue
        name = f"{first} {last}"

        hint = PlayerHint(
            birth_year=parse_birth_year(row.get(args.dob_col)),
            last_team=str(row.get(args.last_team_col, "")).strip() or None,
            position=str(row.get(args.position_col, "")).strip() or None,
        )

        try:
            data = scrape_player(name, hint, leagues_whitelist=args.leagues)
        except Exception as e:
            data = {"name": name, "url": None, "table": [], "error": f"exception: {e}"}

        table = data.get("table") or []
        if not table:
            results.append({
                "Name": name,
                "HDB_URL": data.get("url"),
                "BirthYear": data.get("birth_year"),
                "Position": data.get("position"),
                "Error": data.get("error", data.get("note","no_rows"))
            })
        else:
            for rec in table:
                flat = {
                    "Name": name,
                    "HDB_URL": data.get("url"),
                    "BirthYear": data.get("birth_year"),
                    "Position": data.get("position"),
                    # pass through roster context if helpful
                    "Roster_Last_Team": row.get(args.last_team_col),
                    "Roster_Current_Team": row.get("Current Team"),
                    "Roster_Yr": row.get(args.freshman_col),
                }
                flat.update({f"stat_{k}": v for k, v in rec.items()})
                results.append(flat)

    out = pd.DataFrame(results)
    out.to_csv(args.out, index=False)
    print(f"Saved: {args.out}")

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] --roster ROSTER [--out OUT]
                             [--freshman-col FRESHMAN_COL]
                             [--freshman-values FRESHMAN_VALUES [FRESHMAN_VALUES ...]]
                             [--first-name-col FIRST_NAME_COL]
                             [--last-name-col LAST_NAME_COL]
                             [--position-col POSITION_COL] [--dob-col DOB_COL]
                             [--last-team-col LAST_TEAM_COL]
                             [--leagues LEAGUES [LEAGUES ...]] [--limit LIMIT]
ipykernel_launcher.py: error: ambiguous option: --f=c:\Users\jbanc\AppData\Roaming\jupyter\runtime\kernel-v33ec058fe24e40ce2c302f0f15f2bff59023a1baa.json could match --freshman-col, --freshman-values, --first-name-col


SystemExit: 2

c:\Users\jbanc\anaconda3\envs\data_viz\Lib\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
sample_url = 'https://www.hockeydb.com/ihdb/stats/pdisplay.php?pid=249198'

### Read with pandas
import pandas as pd

dfs = pd.read_html(sample_url)
dfs[0].head()


HTTPError: HTTP Error 403: Forbidden